# Toàn Tập Các Giai Đoạn Dự Án Hybrid OCR

Notebook này tổng hợp toàn bộ vòng đời (lifecycle) của dự án, từ việc sinh dữ liệu, tiền huấn luyện (Pre-training), trích xuất dữ liệu thực tế, tinh chỉnh (Fine-tuning), cho đến kiểm thử mô hình (Inference) từ đầu đến cuối.

## 1. Thiết Lập Môi Trường & Kiểm Tra Phần Cứng
Cài đặt thư viện (nếu cần) và kiểm tra xem hệ thống có nhận diện được GPU (NVIDIA CUDA hoặc Apple MPS) hay không.

In [ ]:
import sys
import os
import torch

# Add project root to path
sys.path.append(os.path.abspath('..'))

if torch.cuda.is_available():
    device = "cuda"
    print(f"NVIDIA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon GPU (MPS) is available.")
else:
    device = "cpu"
    print("Using CPU.")

## 2. Sinh Dữ Liệu Tổng Hợp (Synthetic Data)
Chạy script sinh dữ liệu ngẫu nhiên (chứa các ký tự từ bộ từ vựng chuyên ngành) cùng với các nhiễu giả lập để huấn luyện mô hình cơ sở (Pre-train).

In [ ]:
# Lệnh terminal sinh 50,000 ảnh Train và 5,000 ảnh Val (bỏ qua nếu đã chạy)
!python -m src.hybrid_ocr.dataset.synthetic_data \
    --train-samples 50000 \
    --val-samples 5000 \
    --output-dir ../data \
    --font-dir ../fonts

## 3. Huấn Luyện Pre-training
Huấn luyện mô hình ConvNeXt + Transformer trên tập dữ liệu tổng hợp vừa tạo. Bước này giúp mô hình học cách nhìn mặt chữ tiếng Nhật cơ bản.

In [ ]:
!python -m src.hybrid_ocr.train.train_recognizer \
    --config ../configs/recognition.yaml \
    --train-data ../data/synth_train \
    --val-data ../data/synth_val \
    --stage pretrain \
    --num-workers 4

## 4. Trích Xuất Dữ Liệu Thực Tế (Real Data Extraction)
Sử dụng mô hình PaddleOCR mạnh mẽ để quét file PDF mẫu, nhận dạng và cắt các box chữ (Crops). Tập dữ liệu thực tế này sẽ dùng để Fine-tune.

In [ ]:
!python -m scratch.extract_fine_tune_data \
    --pdf ../pdfs/2026年6月25日-JU愛知-2163-通常車-151-200.pdf \
    --output-dir ../data/real_fine_tune \
    --max-pages 5

## 5. Đồng Bộ Từ Vựng (Vocabulary Sync) & Dọn Dẹp Checkpoint
Để tránh lỗi lệch từ vựng (Mismatch), ta cần sao chép tệp `vocab.json` của mô hình pre-train sang thư mục chuẩn bị fine-tune, và đảm bảo thư mục sạch sẽ.

In [ ]:
!mkdir -p ../runs/finetune
!cp ../runs/recognition/vocab.json ../runs/finetune/vocab.json
!rm -f ../runs/finetune/model_*.pt
print("Đã đồng bộ từ vựng và dọn dẹp thư mục finetune thành công!")

## 6. Tinh Chỉnh Mô Hình (Fine-Tuning)
Chạy tiếp tục việc học nhưng trên tập dữ liệu cắt từ PDF thật. Khóa bộ xương (freeze encoder) và giảm Learning Rate để tránh mất mát kiến thức cũ (Catastrophic Forgetting).

In [ ]:
!python -m src.hybrid_ocr.train.train_recognizer \
    --config ../configs/recognition.yaml \
    --train-data ../data/real_fine_tune \
    --val-data ../data/real_fine_tune \
    --stage finetune \
    --checkpoint ../runs/recognition/model_best.pt \
    --output ../runs/finetune \
    --num-workers 4

## 7. Đánh Giá Toàn Diện (End-to-End Inference)
Khởi tạo hệ thống đường ống hoàn chỉnh (YOLO / Paddle Detector + Mô hình Nhận dạng vừa Fine-tune) và nhận diện trên toàn bộ một trang PDF.

In [ ]:
from src.hybrid_ocr.pipeline import AuctionOCRPipeline
import matplotlib.pyplot as plt
import cv2

# 1. Load pipeline
pipeline = AuctionOCRPipeline(
    yolo_model_path="../yolov8s.pt",
    rec_model_path="../runs/finetune/model_best.pt",
    rec_config_path="../configs/recognition.yaml",
    device=device
)

# 2. Xử lý 1 trang PDF
OUTPUT_DIR = "../runs/inference_demo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
results = pipeline.process_pdf(
    pdf_path="../pdfs/2026年6月25日-JU愛知-2163-通常車-151-200.pdf",
    output_dir=OUTPUT_DIR,
    page_range=(0, 0)
)

# 3. Hiển thị ảnh đầu ra (có vẽ bounding box)
vis_image_path = os.path.join(OUTPUT_DIR, "vis_page_0000.png")
if os.path.exists(vis_image_path):
    img = cv2.imread(vis_image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(15, 20))
    plt.imshow(img)
    plt.axis('off')
    plt.show()